# GNN Applications

## Learning Objectives

1. **Map** GNN outputs to node classification, link prediction, and graph classification tasks
2. **Implement** node classification with a 2-layer GNN
3. **Set up** the link prediction training objective
4. **List** real-world applications across domains

## Three Core Tasks

| Task | Input | Output | Loss |
|------|-------|--------|------|
| **Node classification** | Graph + node features + partial labels | Label for each node | Cross-entropy on labeled nodes |
| **Link prediction** | Graph + node features | Probability of edge existence | Binary cross-entropy on edges/non-edges |
| **Graph classification** | Set of graphs + labels | Graph-level label | Cross-entropy after global pooling |

Each uses GNN to compute node embeddings $h_v$; differs in what is done with them.

In [ ]:
import math, random
from collections import defaultdict

def dot(a, b): return sum(x*y for x,y in zip(a,b))
def sigmoid(x): return 1/(1+math.exp(-max(-20,min(20,x))))
def relu(x): return max(0.0, x)

def mat_vec(W, x):
    return [sum(W[i][j]*x[j] for j in range(len(x))) for i in range(len(W))]

def normalize_adj(adj, n):
    """Compute D^{-1/2} A D^{-1/2} (simplified GCN)."""
    # Add self-loops
    adj_self = defaultdict(set)
    for v in range(n):
        adj_self[v] = adj[v] | {v}
    # Degree with self-loops
    deg = {v: len(adj_self[v]) for v in range(n)}
    return adj_self, deg

def gcn_agg(H, adj_self, deg, W, n):
    """One GCN layer: H_new = D^{-1/2} A_hat D^{-1/2} H W with ReLU."""
    out_dim = len(W)
    new_H = []
    for v in range(n):
        # Aggregate
        agg = [0.0] * len(H[0])
        for u in adj_self[v]:
            norm = 1.0 / math.sqrt(deg[u] * deg[v])
            for i in range(len(H[0])): agg[i] += norm * H[u][i]
        # Linear transform
        new_h = mat_vec(W, agg)
        new_H.append([relu(x) for x in new_h])
    return new_H

# Node classification: 2-layer GCN
edges = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0),(0,3)]
n = 6
adj = defaultdict(set)
for u,v in edges: adj[u].add(v); adj[v].add(u)
adj_self, deg = normalize_adj(adj, n)

rng = random.Random(42)
d_in, d_hidden, n_classes = 3, 4, 2

# Random initial features and weights
H0 = [[rng.gauss(0,1) for _ in range(d_in)] for _ in range(n)]
W1 = [[rng.gauss(0,0.3) for _ in range(d_in)] for _ in range(d_hidden)]
W2 = [[rng.gauss(0,0.3) for _ in range(d_hidden)] for _ in range(n_classes)]

# Forward pass
H1 = gcn_agg(H0, adj_self, deg, W1, n)
H2 = gcn_agg(H1, adj_self, deg, W2, n)
# Softmax for classification
def softmax(v): m=max(v); exps=[math.exp(x-m) for x in v]; s=sum(exps); return [e/s for e in exps]
probs = [softmax(H2[v]) for v in range(n)]

# True labels (assume nodes 0,1,2 = class 0; nodes 3,4,5 = class 1)
labels = [0,0,0,1,1,1]
print("Node classification (2-layer GCN):")
print(f"{'Node':>6} {'P(class 0)':>12} {'P(class 1)':>12} {'Predicted':>10} {'True':>6}")
for v in range(n):
    p = probs[v]
    pred = 0 if p[0]>p[1] else 1
    print(f"{v:>6} {p[0]:>12.4f} {p[1]:>12.4f} {pred:>10} {labels[v]:>6}")

In [ ]:
# Link prediction setup
def link_pred_score(u, v, H):
    """Simple dot product decoder."""
    return sigmoid(dot(H[u], H[v]))

# Positive pairs (existing edges)
pos_pairs = list(edges)
# Negative samples (random non-edges)
all_pairs = [(i,j) for i in range(n) for j in range(i+1,n)]
neg_pairs = [(u,v) for u,v in all_pairs if (u,v) not in edges and (v,u) not in edges][:len(pos_pairs)]

print("Link prediction scores (using last GCN layer H2):")
print(f"{'Pair':>8} {'Score':>8} {'Label':>8}")
for u,v in pos_pairs[:4]:
    s = link_pred_score(u,v,H2)
    print(f"  ({u},{v}):  {s:.4f}  {'1 (edge)':>8}")
for u,v in neg_pairs[:4]:
    s = link_pred_score(u,v,H2)
    print(f"  ({u},{v}):  {s:.4f}  {'0 (no edge)':>8}")
print("
(Scores would improve after training with binary cross-entropy loss)")

## Real-World GNN Applications

| Domain | Task | Graph | Features |
|--------|------|-------|----------|
| **Drug discovery** | Molecular property prediction | Atoms = nodes, bonds = edges | Atom type, bond type |
| **Social networks** | Fake news detection | Users, posts = nodes | Text, follower graph |
| **Citation networks** | Paper classification | Papers = nodes, citations = edges | TF-IDF of abstract |
| **Recommender systems** | Link prediction | Users + items = nodes | Purchase/rating edges |
| **Traffic forecasting** | Speed prediction | Road segments = nodes | Speed, time-of-day |
| **Protein interaction** | Function prediction | Proteins = nodes, interactions = edges | Sequence features |
| **Knowledge graphs** | Relation completion | Entities = nodes, relations = edges | Entity type |

GNNs have become the dominant approach for any task where relational structure matters alongside feature information.